<a href="https://colab.research.google.com/github/srijandubey07-ops/Summer_traning2.0/blob/main/translator_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 AI Translator (NLLB-200)

A free, offline-capable AI translator that supports **200+ languages**, built on Meta's NLLB-200 model via Hugging Face Transformers.

**How to use:** Run each cell below in order (Shift+Enter). The last cell lets you type text and translate it interactively.

> Tip: Go to `Runtime > Change runtime type` and select **T4 GPU** for faster translation (optional, works on CPU too).

## 1. Install dependencies

In [1]:
!pip install -q transformers sentencepiece accelerate

## 2. Load the translation model

We use `facebook/nllb-200-distilled-600M` — a distilled multilingual model that's fast enough to run on Colab's free tier.

In [2]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "facebook/nllb-200-distilled-600M"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print("Model loaded successfully!")

Loading model on: cuda


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully!


## 3. Language codes

NLLB uses FLORES-200 language codes (e.g. `eng_Latn` for English, `hin_Deva` for Hindi). A dictionary of common languages is provided below — add more from the [full FLORES-200 code list](https://github.com/facebookresearch/flores/blob/main/flores200/README.md) if needed.

In [3]:
LANG_CODES = {
    "english": "eng_Latn",
    "hindi": "hin_Deva",
    "spanish": "spa_Latn",
    "french": "fra_Latn",
    "german": "deu_Latn",
    "chinese": "zho_Hans",
    "japanese": "jpn_Jpan",
    "korean": "kor_Hang",
    "arabic": "arb_Arab",
    "russian": "rus_Cyrl",
    "portuguese": "por_Latn",
    "italian": "ita_Latn",
    "bengali": "ben_Beng",
    "urdu": "urd_Arab",
    "tamil": "tam_Taml",
    "telugu": "tel_Telu",
    "punjabi": "pan_Guru",
    "turkish": "tur_Latn",
    "vietnamese": "vie_Latn",
    "dutch": "nld_Latn",
}

print("Available languages:", ", ".join(sorted(LANG_CODES.keys())))

Available languages: arabic, bengali, chinese, dutch, english, french, german, hindi, italian, japanese, korean, portuguese, punjabi, russian, spanish, tamil, telugu, turkish, urdu, vietnamese


## 4. Translation function

In [4]:
def translate(text, source_lang, target_lang):
    """Translate text between two languages using NLLB-200.

    Args:
        text: the text to translate
        source_lang: language name (e.g. 'english') or FLORES code (e.g. 'eng_Latn')
        target_lang: language name or FLORES code
    """
    src_code = LANG_CODES.get(source_lang.lower(), source_lang)
    tgt_code = LANG_CODES.get(target_lang.lower(), target_lang)

    tokenizer.src_lang = src_code
    inputs = tokenizer(text, return_tensors="pt").to(device)

    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_code)

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=512,
    )

    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

## 5. Try it out

In [5]:
sample_text = "Hello, how are you today?"
result = translate(sample_text, "english", "hindi")
print(f"Original ({'english'}): {sample_text}")
print(f"Translated ({'hindi'}): {result}")

Original (english): Hello, how are you today?
Translated (hindi): हैलो, आज आप कैसे हैं?


## 6. Interactive translator

Run this cell and type a sentence plus source/target languages when prompted. Type `exit` to stop.

In [8]:
while True:
    text = input("\nEnter text to translate (or 'exit' to quit): ")
    if text.lower() == "exit":
        break
    src = input("Source language (e.g. english): ")
    if src.lower() == "exit":
        break
    tgt = input("Target language (e.g. hindi): ")
    if tgt.lower() == "exit":
        break

    try:
        output = translate(text, src, tgt)
        print(f"\n>> {output}")
    except Exception as e:
        print(f"Error: {e}. Check your language names against LANG_CODES.")


Enter text to translate (or 'exit' to quit): why
Source language (e.g. english): hindi
Target language (e.g. hindi): hindi

>> क्यों

Enter text to translate (or 'exit' to quit): Srijan
Source language (e.g. english): English
Target language (e.g. hindi): hindi

>> श्रीजन

Enter text to translate (or 'exit' to quit): exit
